## Evaluation

Evaluates Hess Lab IDL, Decode and SMAP result for a single frame by rendering the points in 3D so the user can qualitatively evaluate point correspondence. 

In [32]:
import napari
from numpy import spacing
from skimage.io import imread
import glob
from wip_util import (zero_pad_index, load_iters, 
                      get_np_points, reverse_emitters, 
                      smap_csv_to_emitters, load_iter_results, 
                      print_emitters_info, filter_coordinates,
                      shift_coordinates, filter_sigmas)
import numpy as np
import decode

start_iter = 645
end_iter = 646

frames_per_iter = 250

start_frame = start_iter * frames_per_iter
end_frame = start_frame+100 #end_iter * frames_per_iter

frame_to_view = 50
xmin = 200
xmax = 600
ymin = 200
ymax = 600

In [33]:
# load frames, for this dataset sets of 250 frames are saved in tif sequences of 250 frames.  Each sequence is a single iteration.

iters_path = r'E:\Cryo-PALM Data for Brian\James processing ASCII and RAW Data files\PALM_Run2_slab_0001_488nm_Frames_(Processed_Slab)'
#iters_path = r'/home/bnorthan/janelia_slm/data/Janelia_PALM_RUN2/frames/'
iter_pattern = lambda id:rf'{iters_path}/3DPALM488nm_Iter_{zero_pad_index(id, 4)}_0001_ch0_CAM1_stack0000_405nm_0000000msec_*msecAbs_000x_000y_000z_0001t.tif'
frames = load_iters(start_iter, end_iter, iter_pattern)
print(frames.shape)

loading iter for id: 645
(250, 800, 800)


In [34]:
smap_input_file = r'D:\Janelia_slm_data\Janelia PALM RUN2\SMAP_Result_iter_645.csv'
#smap_input_file = r'/home/bnorthan/janelia_slm/data/Janelia_PALM_RUN2/SMAP/SMAP_Result_iter_645.csv'
smap_emitters = smap_csv_to_emitters(smap_input_file, 100, True)
print('SMAP Emitters',smap_emitters)

hesslab_input_file = r'C:\Users\bnort\work\Janelia_slm\data\James processing ASCII and RAW Data files\2a - PALM_Run2_Slab_0001_488nm_constrained_Gaussian_fitting_gauss_half_width_3_with_winding_file_RAW_UNPROCESSED_PALM_LOCALIZATIONS_ASCII.hdf5'
#hesslab_input_file = r'/home/bnorthan/janelia_slm/data/Janelia_PALM_RUN2/IDL Peak Finder/2a - PALM_Run2_Slab_0001_488nm_constrained_Gaussian_fitting_gauss_half_width_3_with_winding_file_RAW_UNPROCESSED_PALM_LOCALIZATIONS_ASCII.hdf5'
hesslab_emitters=decode.EmitterSet.load(hesslab_input_file)
# filter based on start and end frame
hesslab_emitters = hesslab_emitters[(hesslab_emitters.frame_ix >= start_frame) & (hesslab_emitters.frame_ix < end_frame)]
print('Hesslab Emitters',hesslab_emitters)

decode_iter_path = r'/home/bnorthan/janelia_slm/data/Janelia_PALM_RUN2/processed-2025-06-27'
decode_iter_path = r'D:\Janelia_slm_data\processed\Janelia PALM RUN2\processed-2025-06-27'
decode_emitters = load_iter_results(decode_iter_path, 645, 647)
print('DECODE emitters', decode_emitters)

SMAP Emitters EmitterSet
::num emitters: 37477
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 249
::spanned volume: [    9.036018    11.95644  -1970.      ] - [ 794.8877  795.826  2000.    ]
Hesslab Emitters EmitterSet
::num emitters: 10001
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 161250 - 161349
::spanned volume: [  25.2394     3.28873 -799.951  ] - [798.42  772.354 598.252]


Loading iterations: 100%|██████████| 2/2 [00:00<00:00, 60.62it/s]

DECODE emitters EmitterSet
::num emitters: 345830
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 161250 - 161749
::spanned volume: [-4.5627666e-01 -6.3932061e-01 -1.2588881e+03] - [ 799.507   799.4638 1246.2266]


In [35]:
decode_emitters.frame_ix = decode_emitters.frame_ix - start_frame 
hesslab_emitters.frame_ix = hesslab_emitters.frame_ix - start_frame

print('SMAP Emitters')
print_emitters_info(smap_emitters)
print('Hesslab Emitters')
print_emitters_info(hesslab_emitters)
print('DECODE Emitters')
print_emitters_info(decode_emitters)

SMAP Emitters
EmitterSet
::num emitters: 37477
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 249
::spanned volume: [    9.036018    11.95644  -1970.      ] - [ 794.8877  795.826  2000.    ]
x sig range (nm) 1.95 1040.07
y sig range (nm) 1.95 1040.07
z sig range (nm) 5.69 3014.08

Hesslab Emitters
EmitterSet
::num emitters: 10001
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 99
::spanned volume: [  25.2394     3.28873 -799.951  ] - [798.42  772.354 598.252]
x sig range (nm) 143.46 295.20
y sig range (nm) 135.31 253.91
z sig range (nm) 3.58 249.26

DECODE Emitters
EmitterSet
::num emitters: 345830
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 499
::spanned volume: [-4.5627666e-01 -6.3932061e-01 -1.2588881e+03] - [ 799.507   799.4638 1246.2266]
x sig range (nm) 1.41 382.21
y sig range (nm) 1.52 366.36
z sig range (nm) 6.42 3746.29



In [36]:
print(decode_emitters)
decode_emitters = filter_coordinates(decode_emitters, xmin = xmin, xmax=xmax, ymin=ymin, ymax=ymax, zmin=-800, zmax=800)
smap_emitters = filter_coordinates(smap_emitters, xmin = xmin, xmax=xmax, ymin=ymin, ymax=ymax, zmin=-800, zmax=800)
hesslab_emitters = filter_coordinates(hesslab_emitters, xmin = xmin, xmax=xmax, ymin=ymin, ymax=ymax, zmin=-800, zmax=800)
print(decode_emitters)

decode_emitters = shift_coordinates(decode_emitters, -xmin, -ymin, 0)
smap_emitters = shift_coordinates(smap_emitters, -xmin, -ymin, 0)
hesslab_emitters = shift_coordinates(hesslab_emitters, -xmin, -ymin, 0)

EmitterSet
::num emitters: 345830
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 499
::spanned volume: [-4.5627666e-01 -6.3932061e-01 -1.2588881e+03] - [ 799.507   799.4638 1246.2266]
EmitterSet
::num emitters: 49015
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 499
::spanned volume: [ 200.98706  200.35886 -799.95026] - [599.9868  599.98975 799.94165]


In [37]:
decode_emitters.phot.min(), decode_emitters.phot.max()

(tensor(125.1813), tensor(4775.0830))

In [38]:
print(decode_emitters)
decode_emitters = filter_sigmas(decode_emitters, x_sig_max = 100, y_sig_max = 100, z_sig_max = 300, prob_min=0.50, phot_min=0)
smap_emitters = filter_sigmas(smap_emitters, x_sig_max = 100, y_sig_max = 100, z_sig_max = 300, prob_min=0.0, phot_min=0)
hesslab_emitters = filter_sigmas(hesslab_emitters, x_sig_max = 500, y_sig_max = 500, z_sig_max = 1000, prob_min=0.0, phot_min=0)
print(decode_emitters)

EmitterSet
::num emitters: 49015
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 499
::spanned volume: [ 9.8706055e-01  3.5885620e-01 -7.9995026e+02] - [399.98682 399.98975 799.94165]
EmitterSet
::num emitters: 37372
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 499
::spanned volume: [   1.0055084    1.4617767 -799.8079   ] - [399.9751  399.98975 799.94165]


In [39]:
viewer = napari.Viewer()
viewer.add_image(frames[frame_to_view, ymin:ymax, xmin:xmax])

<Image layer 'Image' at 0x2100d50d700>

In [40]:
decode_points = get_np_points(decode_emitters, 0, 100, True)
print(decode_points.shape)
decode_points = decode_points[decode_points[:,0] == frame_to_view]
print(decode_points.shape)
decode_points_t = decode_points[:,1:4]
decode_points_t = decode_points_t[:, [2, 0, 1]]  # Reorder to (z, y, x)

print(smap_emitters)
smap_points = get_np_points(smap_emitters, 0, 100, True)
print(smap_points.shape)
smap_points = smap_points[smap_points[:,0] == frame_to_view]
print(smap_points.shape)
smap_points_t = smap_points[:,1:4]
smap_points_t = smap_points_t[:, [2, 0, 1]]  # Reorder to (z, y, x)

print(hesslab_emitters)
hesslab_points = get_np_points(hesslab_emitters, 0, 100, True)
print(hesslab_points.shape)

hesslab_points = hesslab_points[hesslab_points[:,0] == frame_to_view]
print(hesslab_points.shape)
hesslab_points_t = hesslab_points[:,1:4]
hesslab_points_t = hesslab_points_t[:, [2, 1, 0]]  # Reorder to (z, y, x)


(7538, 4)
(79, 4)
EmitterSet
::num emitters: 12542
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 249
::spanned volume: [ 5.340000e-02  1.934100e+00 -7.958517e+02] - [399.6788 395.2722 799.6304]
(5002, 4)
(54, 4)
EmitterSet
::num emitters: 2853
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 99
::spanned volume: [   1.168   24.03  -799.951] - [377.933 384.261 596.247]
(2853, 4)
(30, 4)


In [41]:
#decode_points_t[:,0] = decode_points_t[:,0] - decode_points_t[:,0].min()
#hesslab_points_t[:,0] = hesslab_points_t[:,0] - hesslab_points_t[:,0].min()
hesslab_points_t[:,0] = - hesslab_points_t[:,0]  # invert z axis for hesslab points

In [42]:
hesslab_points_t[:,0].min(), hesslab_points_t[:,0].max()

(-376.698, 790.287)

In [43]:
scale = [1, 1, 1]
size = 3
viewer.add_points(decode_points_t, size=size, face_color='blue', name='decode emitters (t)', scale=scale)
viewer.add_points(hesslab_points_t, size=size, face_color='red', name='hesslab emitters (t)', scale=scale)
viewer.add_points(smap_points_t, size=size, face_color='green', name='smap emitters (t)', scale=scale)

<Points layer 'smap emitters (t)' at 0x21010ea1070>

In [29]:
decode_points_t

array([[ 630.66473389,   23.84719849,  101.14129639],
       [ 697.6998291 ,   57.55963135,  117.73583984],
       [ 582.80914307,   86.07888794,  204.87988281],
       [ 777.62670898,  112.28723145,  214.17565918],
       [ 460.76599121,  112.47302246,  376.86157227],
       [ 569.95214844,  114.49752808,  116.5921936 ],
       [ 445.81430054,  116.83825684,  211.88821411],
       [ 575.92681885,  130.48452759,  194.56335449],
       [ 112.41125488,  131.82962036,  156.80654907],
       [ 791.03131104,  135.8894043 ,  116.26269531],
       [-552.05078125,  139.0473938 ,  183.36123657],
       [ 201.81272888,  140.64303589,  210.05938721],
       [ 663.89788818,  144.7701416 ,  319.13824463],
       [ -43.81967926,  146.62042236,  141.12341309],
       [ 210.94235229,  147.18511963,  164.96557617],
       [ 218.99180603,  146.668396  ,  258.93588257],
       [ 617.4296875 ,  150.34448242,   99.89239502],
       [ 335.24450684,  150.3762207 ,  182.00811768],
       [ 170.75982666,  150.

In [30]:
print(decode_points_t[:,0].min(), decode_points_t[:,0].max(), decode_points_t[:,1].min(), decode_points_t[:,1].max(), decode_points_t[:,2].min(), decode_points_t[:,2].max())
print(hesslab_points_t[:,0].min(), hesslab_points_t[:,0].max(), hesslab_points_t[:,1].min(), hesslab_points_t[:,1].max(), hesslab_points_t[:,2].min(), hesslab_points_t[:,2].max())
print(smap_points_t[:,0].min(), smap_points_t[:,0].max(), smap_points_t[:,1].min(), smap_points_t[:,1].max(), smap_points_t[:,2].min(), smap_points_t[:,2].max())

-789.635986328125 791.0313110351562 23.847198486328125 395.1981201171875 5.353607177734375 376.861572265625
-376.698 790.287 24.126000000000005 377.64700000000005 95.58999999999997 377.80499999999995
-713.6487 779.7497 24.947000000000003 375.63710000000003 6.121100000000013 377.8276000000001


In [31]:
print(hesslab_emitters)
hesslab_points = get_np_points(hesslab_emitters, start_frame, start_frame+100, True)
hesslab_points.shape
hesslab_points[:,0].max(), hesslab_points[:,1].max(), hesslab_points[:,2].max(), hesslab_points[:,3].max()

EmitterSet
::num emitters: 2853
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 99
::spanned volume: [   1.168   24.03  -799.951] - [377.933 384.261 596.247]


ValueError: zero-size array to reduction operation maximum which has no identity

In [ ]:
decode_points_t[:,0].min(), decode_points_t[:,1].min(), decode_points_t[:,2].min()

(-819.17822265625, 31.8754940032959, 41.91632080078125)

In [ ]:

hesslab_points_t[:,0].max(), hesslab_points_t[:,1].max(), hesslab_points_t[:,2].max()

(508.158, 793.857, 692.392)